# Spatial dual-bond local identity — symbolic gate

This fail-closed notebook runs one preregistered symbolic judge at one immutable raw SHA. It licenses only the local 2×2 factorization, not a many-site or sector theorem.

In [ ]:
import datetime, hashlib, json, os, platform, subprocess, sys, tempfile, urllib.request
from pathlib import Path

RAW_COMMIT = 'cea31d99abee7d6b93ff8bebc551859c687da716'
RAW_URL = ('https://raw.githubusercontent.com/lluiseriksson/'
    f'THE-ERIKSSON-PROGRAMME/{RAW_COMMIT}/scripts/judge_spatial_dual_bond.py')
EXPECTED_SHA256 = 'bd0fda3a06b4d52bb4dd5d2230d4520d0d5ef2978ca293cbf82e963fbf61199d'

run_root = Path(tempfile.mkdtemp(prefix='spatial-dual-bond-'))
judge = run_root / 'judge_spatial_dual_bond.py'
source = urllib.request.urlopen(RAW_URL, timeout=60).read()
source_hash = hashlib.sha256(source).hexdigest()
if source_hash != EXPECTED_SHA256:
    raise RuntimeError(f'judge SHA-256 mismatch: {source_hash}')
judge.write_bytes(source)

transcript = {
    'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'runtime': platform.platform(),
    'python': platform.python_version(),
    'cpu_count': os.cpu_count(),
    'raw_commit': RAW_COMMIT,
    'judge_sha256': source_hash,
    'runs': [],
}
for flags in ([], ['-O']):
    command = [sys.executable, *flags, str(judge)]
    result = subprocess.run(command, text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, check=False)
    print('$', ' '.join(command))
    print(result.stdout, end='')
    print(f'[exit {result.returncode}]')
    transcript['runs'].append({'flags': flags, 'exit': result.returncode,
        'output': result.stdout})
    if result.returncode != 0 or '"status": "PASS"' not in result.stdout:
        raise RuntimeError(f'judge failed under flags {flags}')

artifact = run_root / 'dual_bond_gate.json'
artifact.write_text(json.dumps(transcript, indent=2, sort_keys=True) + '\n',
    encoding='utf-8')
artifact_hash = hashlib.sha256(artifact.read_bytes()).hexdigest()
print(f'artifact_sha256={artifact_hash}')
print('SPATIAL DUAL-BOND LOCAL GATE PASS')
from google.colab import files
files.download(str(artifact))